# LoRA Fine-Tuning for CLIP Retrieval

This notebook gives you a **LoRA-based fine-tuning pipeline** for image-text retrieval using:

- `transformers.CLIPModel`
- `peft` LoRA adapters
- multi-positive contrastive loss
- epoch-wise validation Recall@1 / Recall@5 / Recall@10
- base submission
- reranked submission
- ensemble submission

## Why this uses Hugging Face CLIP

LoRA support is much cleaner with:

- `transformers.CLIPModel`
- `peft.LoraConfig`
- `get_peft_model(...)`

So this notebook is the best way to test LoRA properly.

In [6]:
# Install if needed
!pip install -q accelerate peft

In [7]:
import os
import re
import math
import time
import random
import unicodedata
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import CLIPModel, CLIPProcessor, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model

In [8]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

In [26]:
@dataclass
class CFG:
    DATA_DIR: Path = Path("./data/")
    TRAIN_CSV: Path = DATA_DIR / "train.csv"
    VAL_CSV: Path = DATA_DIR / "val.csv"
    TEST_CSV: Path = DATA_DIR / "test.csv"
    CAPTION_POOL_CSV: Path = DATA_DIR / "caption_pool.csv"
    IMAGE_POOL_CSV: Path = DATA_DIR / "image_pool.csv"
    SAMPLE_SUBMISSION_CSV: Path = DATA_DIR / "sample_submission.csv"

    TRAIN_IMAGE_DIR: Path = DATA_DIR / "train_images/train_images"
    VAL_IMAGE_DIR: Path = DATA_DIR / "val_images/val_images"
    TEST_IMAGE_DIR: Path = DATA_DIR / "test_images/test_images"

    OUTPUT_DIR: str = "./outputs_lora_clip"

    MODEL_NAME: str = "openai/clip-vit-base-patch16"
    MAX_TEXT_LEN: int = 77

    LORA_R: int = 8
    LORA_ALPHA: int = 16
    LORA_DROPOUT: float = 0.05
    TARGET_MODULES: tuple = ("q_proj", "k_proj", "v_proj", "out_proj")

    EPOCHS: int = 5
    BATCH_SIZE: int = 32
    NUM_WORKERS: int = 0
    LR: float = 2e-4
    WEIGHT_DECAY: float = 1e-4
    WARMUP_RATIO: float = 0.05
    ACCUM_STEPS: int = 1
    LABEL_SMOOTHING: float = 0.0

    UNFREEZE_LOGIT_SCALE: bool = True
    EMBED_BATCH_SIZE: int = 128
    ENSEMBLE_CKPTS: int = 3
    SAVE_EACH_EPOCH: bool = True

    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = torch.cuda.is_available()

cfg = CFG()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
print(cfg)

CFG(DATA_DIR=PosixPath('data'), TRAIN_CSV=PosixPath('data/train.csv'), VAL_CSV=PosixPath('data/val.csv'), TEST_CSV=PosixPath('data/test.csv'), CAPTION_POOL_CSV=PosixPath('data/caption_pool.csv'), IMAGE_POOL_CSV=PosixPath('data/image_pool.csv'), SAMPLE_SUBMISSION_CSV=PosixPath('data/sample_submission.csv'), TRAIN_IMAGE_DIR=PosixPath('data/train_images/train_images'), VAL_IMAGE_DIR=PosixPath('data/val_images/val_images'), TEST_IMAGE_DIR=PosixPath('data/test_images/test_images'), OUTPUT_DIR='./outputs_lora_clip', MODEL_NAME='openai/clip-vit-base-patch16', MAX_TEXT_LEN=77, LORA_R=8, LORA_ALPHA=16, LORA_DROPOUT=0.05, TARGET_MODULES=('q_proj', 'k_proj', 'v_proj', 'out_proj'), EPOCHS=5, BATCH_SIZE=32, NUM_WORKERS=0, LR=0.0002, WEIGHT_DECAY=0.0001, WARMUP_RATIO=0.05, ACCUM_STEPS=1, LABEL_SMOOTHING=0.0, UNFREEZE_LOGIT_SCALE=True, EMBED_BATCH_SIZE=128, ENSEMBLE_CKPTS=3, SAVE_EACH_EPOCH=True, DEVICE='cuda', USE_AMP=True)


In [27]:
train_df = pd.read_csv(cfg.TRAIN_CSV)
val_df = pd.read_csv(cfg.VAL_CSV)
test_df = pd.read_csv(cfg.TEST_CSV)
caption_pool_df = pd.read_csv(cfg.CAPTION_POOL_CSV)
image_pool_df = pd.read_csv(cfg.IMAGE_POOL_CSV)
sample_submission_df = pd.read_csv(cfg.SAMPLE_SUBMISSION_CSV)

display(train_df.head())
display(val_df.head())
display(caption_pool_df.head())
display(image_pool_df.head())
display(test_df.head())

,image_id,caption_id
0,1,1
1,1,2
2,1,3
3,1,4
4,1,5


,image_id,caption_id
0,8001,40001
1,8001,40002
2,8001,40003
3,8001,40004
4,8001,40005


,caption_id,image_id,caption
0,1,1,Two young guys with shaggy hair look at their ...
1,2,1,"Two young , White males are outside near many ..."
2,3,1,Two men in green shirts are standing in a yard .
3,4,1,A man in a blue shirt standing in a garden .
4,5,1,Two friends enjoy time spent together .


,image_id,image_name
0,1,1000092795.jpg
1,2,10002456.jpg
2,3,1000268201.jpg
3,4,1000344755.jpg
4,5,1000366164.jpg


,id,query,query_type
0,1,9422,image
1,2,9880,image
2,3,9601,image
3,4,9511,image
4,5,9324,image


## Better text handling

In [28]:
def normalize_caption(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

caption_pool_df["caption"] = caption_pool_df["caption"].apply(normalize_caption)

In [29]:
image_id_to_name = dict(zip(image_pool_df["image_id"], image_pool_df["image_name"]))
caption_id_to_text = dict(zip(caption_pool_df["caption_id"], caption_pool_df["caption"]))
caption_id_to_image_id = dict(zip(caption_pool_df["caption_id"], caption_pool_df["image_id"]))
image_id_to_caption_ids = caption_pool_df.groupby("image_id")["caption_id"].apply(list).to_dict()

print("num images:", len(image_id_to_name))
print("num captions:", len(caption_id_to_text))

num images: 10000
num captions: 50000


In [30]:
processor = CLIPProcessor.from_pretrained(cfg.MODEL_NAME)
tokenizer = processor.tokenizer

In [31]:
class PairDataset(Dataset):
    def __init__(self, pairs_df, image_dir, image_id_to_name, caption_id_to_text):
        self.df = pairs_df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.image_id_to_name = image_id_to_name
        self.caption_id_to_text = caption_id_to_text

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = int(row["image_id"])
        caption_id = int(row["caption_id"])
        image_name = self.image_id_to_name[image_id]
        image_path = self.image_dir / image_name
        image = Image.open(image_path).convert("RGB")
        text = normalize_caption(self.caption_id_to_text.get(caption_id, ""))
        return {
            "image": image,
            "text": text,
            "image_id": image_id,
            "caption_id": caption_id,
        }

def collate_fn(batch):
    images = [x["image"] for x in batch]
    texts = [x["text"] for x in batch]
    image_ids = torch.tensor([x["image_id"] for x in batch], dtype=torch.long)
    caption_ids = torch.tensor([x["caption_id"] for x in batch], dtype=torch.long)

    proc = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=cfg.MAX_TEXT_LEN,
    )
    proc["image_ids"] = image_ids
    proc["caption_ids"] = caption_ids
    proc["raw_texts"] = texts
    return proc

In [32]:
train_dataset = PairDataset(train_df, cfg.TRAIN_IMAGE_DIR, image_id_to_name, caption_id_to_text)
val_dataset = PairDataset(val_df, cfg.VAL_IMAGE_DIR, image_id_to_name, caption_id_to_text)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))

train batches: 1250
val batches: 157


## CLIP + LoRA

In [33]:
base_model = CLIPModel.from_pretrained(cfg.MODEL_NAME)

lora_config = LoraConfig(
    r=cfg.LORA_R,
    lora_alpha=cfg.LORA_ALPHA,
    lora_dropout=cfg.LORA_DROPOUT,
    bias="none",
    target_modules=list(cfg.TARGET_MODULES),
)

model = get_peft_model(base_model, lora_config)
if not cfg.UNFREEZE_LOGIT_SCALE:
    model.logit_scale.requires_grad = False

model = model.to(cfg.DEVICE)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 983,040 || all params: 150,603,777 || trainable%: 0.6527


## Multi-positive loss

This handles the fact that one image can have multiple valid captions.

In [34]:
def multi_positive_clip_loss(image_embeds, text_embeds, image_ids, temperature, label_smoothing=0.0):
    image_embeds = F.normalize(image_embeds, dim=-1)
    text_embeds = F.normalize(text_embeds, dim=-1)

    logits = image_embeds @ text_embeds.T
    logits = logits * torch.exp(temperature)

    pos_mask = (image_ids[:, None] == image_ids[None, :]).float()

    targets_i2t = pos_mask / pos_mask.sum(dim=1, keepdim=True).clamp_min(1.0)
    targets_t2i = targets_i2t.T

    if label_smoothing > 0:
        n = targets_i2t.size(1)
        smooth = label_smoothing / n
        targets_i2t = (1 - label_smoothing) * targets_i2t + smooth
        targets_t2i = (1 - label_smoothing) * targets_t2i + smooth

    log_probs_i2t = F.log_softmax(logits, dim=1)
    log_probs_t2i = F.log_softmax(logits.T, dim=1)

    loss_i2t = -(targets_i2t * log_probs_i2t).sum(dim=1).mean()
    loss_t2i = -(targets_t2i * log_probs_t2i).sum(dim=1).mean()
    return 0.5 * (loss_i2t + loss_t2i), logits

In [35]:
def get_image_text_embeds(model, batch):
    tensor_batch = {k: v.to(cfg.DEVICE) for k, v in batch.items() if isinstance(v, torch.Tensor)}
    outputs = model(
        input_ids=tensor_batch["input_ids"],
        attention_mask=tensor_batch["attention_mask"],
        pixel_values=tensor_batch["pixel_values"],
        return_loss=False,
    )
    return outputs.image_embeds, outputs.text_embeds

In [36]:
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=cfg.LR,
    weight_decay=cfg.WEIGHT_DECAY,
)

total_train_steps = math.ceil(len(train_loader) / cfg.ACCUM_STEPS) * cfg.EPOCHS
warmup_steps = int(total_train_steps * cfg.WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_train_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=cfg.USE_AMP)
print("total_train_steps:", total_train_steps)
print("warmup_steps:", warmup_steps)

total_train_steps: 6250
warmup_steps: 312


/tmp/ipykernel_1564516/261337684.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg.USE_AMP)


In [37]:
def train_one_epoch(model, loader, optimizer, scaler, device, epoch, total_epochs):
    model.train()
    running_loss = 0.0
    start_time = time.time()
    progress = tqdm(loader, desc=f"Train [{epoch}/{total_epochs}]", leave=False, position=1)
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(progress, start=1):
        image_ids = batch["image_ids"].to(device)

        with torch.cuda.amp.autocast(enabled=cfg.USE_AMP):
            image_embeds, text_embeds = get_image_text_embeds(model, batch)
            loss, _ = multi_positive_clip_loss(
                image_embeds,
                text_embeds,
                image_ids,
                model.logit_scale,
                cfg.LABEL_SMOOTHING,
            )
            loss = loss / cfg.ACCUM_STEPS

        scaler.scale(loss).backward()

        if step % cfg.ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        running_loss += loss.item() * cfg.ACCUM_STEPS
        progress.set_postfix(loss=f"{running_loss / step:.4f}")

    return running_loss / max(len(loader), 1), time.time() - start_time


@torch.no_grad()
def valid_one_epoch(model, loader, device, epoch, total_epochs):
    model.eval()
    running_loss = 0.0
    start_time = time.time()
    progress = tqdm(loader, desc=f"Valid [{epoch}/{total_epochs}]", leave=False, position=1)

    for step, batch in enumerate(progress, start=1):
        image_ids = batch["image_ids"].to(device)
        with torch.cuda.amp.autocast(enabled=cfg.USE_AMP):
            image_embeds, text_embeds = get_image_text_embeds(model, batch)
            loss, _ = multi_positive_clip_loss(
                image_embeds,
                text_embeds,
                image_ids,
                model.logit_scale,
                0.0,
            )

        running_loss += loss.item()
        progress.set_postfix(loss=f"{running_loss / step:.4f}")

    return running_loss / max(len(loader), 1), time.time() - start_time

## Retrieval helpers

In [38]:
class ImageOnlyDataset(Dataset):
    def __init__(self, image_ids, image_dir, image_id_to_name):
        self.image_ids = list(image_ids)
        self.image_dir = Path(image_dir)
        self.image_id_to_name = image_id_to_name

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = int(self.image_ids[idx])
        image_name = self.image_id_to_name[image_id]
        image = Image.open(self.image_dir / image_name).convert("RGB")
        return {"image_id": image_id, "image": image}

class TextOnlyDataset(Dataset):
    def __init__(self, caption_ids, caption_id_to_text):
        self.caption_ids = list(caption_ids)
        self.caption_id_to_text = caption_id_to_text

    def __len__(self):
        return len(self.caption_ids)

    def __getitem__(self, idx):
        caption_id = int(self.caption_ids[idx])
        text = normalize_caption(self.caption_id_to_text[caption_id])
        return {"caption_id": caption_id, "text": text}

def image_collate_fn(batch):
    images = [x["image"] for x in batch]
    image_ids = [x["image_id"] for x in batch]
    proc = processor(images=images, return_tensors="pt")
    proc["image_ids"] = image_ids
    return proc

def text_collate_fn(batch):
    texts = [x["text"] for x in batch]
    caption_ids = [x["caption_id"] for x in batch]
    proc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=cfg.MAX_TEXT_LEN,
        return_tensors="pt",
    )
    proc["caption_ids"] = caption_ids
    return proc

In [43]:
@torch.no_grad()
def compute_image_embeddings(model, image_ids, image_dir):
    dataset = ImageOnlyDataset(image_ids, image_dir, image_id_to_name)
    loader = DataLoader(
        dataset,
        batch_size=cfg.EMBED_BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        collate_fn=image_collate_fn,
    )

    all_ids, all_embs = [], []
    model.eval()

    for batch in tqdm(loader, desc="Encode images", leave=False):
        pixel_values = batch["pixel_values"].to(cfg.DEVICE)

        # use CLIP vision encoder directly
        vision_outputs = model.base_model.vision_model(pixel_values=pixel_values)
        pooled_output = vision_outputs.pooler_output
        embs = model.base_model.visual_projection(pooled_output)

        embs = F.normalize(embs, dim=-1)

        all_ids.extend(batch["image_ids"])
        all_embs.append(embs.cpu())

    return all_ids, torch.cat(all_embs, dim=0)

@torch.no_grad()
def compute_text_embeddings(model, caption_ids):
    dataset = TextOnlyDataset(caption_ids, caption_id_to_text)
    loader = DataLoader(
        dataset,
        batch_size=cfg.EMBED_BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        collate_fn=text_collate_fn,
    )

    all_ids, all_embs = [], []
    model.eval()

    for batch in tqdm(loader, desc="Encode texts", leave=False):
        input_ids = batch["input_ids"].to(cfg.DEVICE)
        attention_mask = batch["attention_mask"].to(cfg.DEVICE)

        text_outputs = model.base_model.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        pooled_output = text_outputs.pooler_output
        embs = model.base_model.text_projection(pooled_output)

        embs = F.normalize(embs, dim=-1)

        all_ids.extend(batch["caption_ids"])
        all_embs.append(embs.cpu())

    return all_ids, torch.cat(all_embs, dim=0)

In [44]:
def recall_at_k_image_to_text(image_ids, retrieved_caption_ids, image_id_to_caption_ids, k):
    hits = 0
    for img_id, preds in zip(image_ids, retrieved_caption_ids):
        gt = set(image_id_to_caption_ids[img_id])
        if any(p in gt for p in preds[:k]):
            hits += 1
    return hits / len(image_ids)

def recall_at_k_text_to_image(caption_ids, retrieved_image_ids, caption_id_to_image_id, k):
    hits = 0
    for cap_id, preds in zip(caption_ids, retrieved_image_ids):
        gt = caption_id_to_image_id[cap_id]
        if gt in preds[:k]:
            hits += 1
    return hits / len(caption_ids)

In [45]:
@torch.no_grad()
def evaluate_retrieval(model):
    val_image_ids = sorted(val_df["image_id"].unique().tolist())
    val_caption_ids = sorted(val_df["caption_id"].unique().tolist())

    image_ids, image_embs = compute_image_embeddings(model, val_image_ids, cfg.VAL_IMAGE_DIR)
    caption_ids, caption_embs = compute_text_embeddings(model, val_caption_ids)

    sim_i2t = image_embs @ caption_embs.T
    top10_i2t = torch.topk(sim_i2t, k=10, dim=1).indices.numpy()
    retrieved_caption_ids = [[caption_ids[j] for j in row] for row in top10_i2t]

    i2t_r1 = recall_at_k_image_to_text(image_ids, retrieved_caption_ids, image_id_to_caption_ids, 1)
    i2t_r5 = recall_at_k_image_to_text(image_ids, retrieved_caption_ids, image_id_to_caption_ids, 5)
    i2t_r10 = recall_at_k_image_to_text(image_ids, retrieved_caption_ids, image_id_to_caption_ids, 10)
    i2t_score = (i2t_r1 + i2t_r5 + i2t_r10) / 3.0

    sim_t2i = caption_embs @ image_embs.T
    top10_t2i = torch.topk(sim_t2i, k=10, dim=1).indices.numpy()
    retrieved_image_ids = [[image_ids[j] for j in row] for row in top10_t2i]

    t2i_r1 = recall_at_k_text_to_image(caption_ids, retrieved_image_ids, caption_id_to_image_id, 1)
    t2i_r5 = recall_at_k_text_to_image(caption_ids, retrieved_image_ids, caption_id_to_image_id, 5)
    t2i_r10 = recall_at_k_text_to_image(caption_ids, retrieved_image_ids, caption_id_to_image_id, 10)
    t2i_score = (t2i_r1 + t2i_r5 + t2i_r10) / 3.0

    return {
        "i2t_r1": i2t_r1,
        "i2t_r5": i2t_r5,
        "i2t_r10": i2t_r10,
        "i2t_score": i2t_score,
        "t2i_r1": t2i_r1,
        "t2i_r5": t2i_r5,
        "t2i_r10": t2i_r10,
        "t2i_score": t2i_score,
        "mean_score": (i2t_score + t2i_score) / 2.0,
    }

In [46]:
history = []
best_mean_score = -1.0
best_ckpt_path = Path(cfg.OUTPUT_DIR) / "best_lora_clip_model.pt"

overall_start = time.time()
outer_pbar = tqdm(range(cfg.EPOCHS), desc="Overall Training Progress", position=0)

for epoch_idx in outer_pbar:
    epoch = epoch_idx + 1
    print(f"\n{'='*72}")
    print(f"Epoch {epoch}/{cfg.EPOCHS}")
    print(f"{'='*72}")

    train_loss, train_time = train_one_epoch(model, train_loader, optimizer, scaler, cfg.DEVICE, epoch, cfg.EPOCHS)
    val_loss, val_time = valid_one_epoch(model, val_loader, cfg.DEVICE, epoch, cfg.EPOCHS)
    metrics = evaluate_retrieval(model)
    mean_score = metrics["mean_score"]

    improved = mean_score > best_mean_score
    if improved:
        best_mean_score = mean_score
        torch.save({
            "model": model.state_dict(),
            "epoch": epoch,
            "metrics": metrics,
            "cfg": asdict(cfg),
        }, best_ckpt_path)

    if cfg.SAVE_EACH_EPOCH:
        torch.save({
            "model": model.state_dict(),
            "epoch": epoch,
            "metrics": metrics,
            "cfg": asdict(cfg),
        }, Path(cfg.OUTPUT_DIR) / f"lora_clip_epoch_{epoch}.pt")

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_time_sec": train_time,
        "val_time_sec": val_time,
        "is_best": improved,
        **metrics,
    })

    outer_pbar.set_postfix(
        train_loss=f"{train_loss:.4f}",
        val_loss=f"{val_loss:.4f}",
        mean_score=f"{mean_score:.4f}",
        best=f"{best_mean_score:.4f}"
    )

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"I2T Score  : {metrics['i2t_score']:.4f} (R@1={metrics['i2t_r1']:.4f}, R@5={metrics['i2t_r5']:.4f}, R@10={metrics['i2t_r10']:.4f})")
    print(f"T2I Score  : {metrics['t2i_score']:.4f} (R@1={metrics['t2i_r1']:.4f}, R@5={metrics['t2i_r5']:.4f}, R@10={metrics['t2i_r10']:.4f})")
    print(f"Mean Score : {metrics['mean_score']:.4f}")
    if improved:
        print("Best model updated.")

total_time = time.time() - overall_start
history_df = pd.DataFrame(history)
display(history_df)
print("Best checkpoint:", best_ckpt_path)
print(f"Total training time: {total_time/60:.2f} minutes")

Overall Training Progress:   0%|          | 0/5 [00:00<?, ?it/s]


Epoch 1/5


Train [1/5]:   0%|          | 0/1250 [00:00<?, ?it/s]

/tmp/ipykernel_1564516/3816903250.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=cfg.USE_AMP):


Valid [1/5]:   0%|          | 0/157 [00:00<?, ?it/s]

...

Encode images:   0%|          | 0/8 [00:00<?, ?it/s]

Encode texts:   0%|          | 0/40 [00:00<?, ?it/s]

Train Loss : 0.0250
Val Loss   : 2.4539
I2T Score  : 0.9557 (R@1=0.8850, R@5=0.9840, R@10=0.9980)
T2I Score  : 0.8939 (R@1=0.7582, R@5=0.9464, R@10=0.9770)
Mean Score : 0.9248
Best model updated.


,epoch,train_loss,val_loss,train_time_sec,val_time_sec,is_best,i2t_r1,i2t_r5,i2t_r10,i2t_score,t2i_r1,t2i_r5,t2i_r10,t2i_score,mean_score
0,1,0.051908,2.341295,353.490495,35.475651,True,0.891,0.987,0.998,0.958667,0.7456,0.9442,0.9758,0.888533,0.923600
1,2,0.033992,2.429331,371.779758,34.821849,False,0.881,0.981,0.996,0.952667,0.7520,0.9444,0.9760,0.890800,0.921733
2,3,0.027990,2.443717,363.521744,38.156758,True,0.884,0.985,0.998,0.955667,0.7574,0.9444,0.9756,0.892467,0.924067
3,4,0.025171,2.449178,378.568856,42.755876,True,0.888,0.986,0.998,0.957333,0.7562,0.9442,0.9758,0.892067,0.924700
4,5,0.025001,2.453887,356.240280,34.522771,True,0.885,0.984,0.998,0.955667,0.7582,0.9464,0.9770,0.893867,0.924767


Best checkpoint: outputs_lora_clip/best_lora_clip_model.pt
Total training time: 34.61 minutes


In [48]:
ckpt = torch.load(best_ckpt_path, map_location=cfg.DEVICE, weights_only=False)
model.load_state_dict(ckpt["model"])
model.eval()
print("Loaded best checkpoint:", best_ckpt_path)
print("Metrics:", ckpt["metrics"])

Loaded best checkpoint: outputs_lora_clip/best_lora_clip_model.pt
Metrics: {'i2t_r1': 0.885, 'i2t_r5': 0.984, 'i2t_r10': 0.998, 'i2t_score': 0.9556666666666667, 't2i_r1': 0.7582, 't2i_r5': 0.9464, 't2i_r10': 0.977, 't2i_score': 0.8938666666666667, 'mean_score': 0.9247666666666667}


## Base submission

In [51]:
@torch.no_grad()
def build_base_submission(model):
    from pathlib import Path

    model.eval()

    # --------------------------------------------------
    # 1. Caption candidate pool
    # --------------------------------------------------
    candidate_caption_ids = caption_pool_df["caption_id"].astype(int).tolist()
    cap_ids, cap_embs = compute_text_embeddings(model, candidate_caption_ids)

    # --------------------------------------------------
    # 2. Image candidate pool
    #    Keep only images that actually exist in test_images
    # --------------------------------------------------
    candidate_image_ids = []

    for _, row in image_pool_df.iterrows():
        image_id = int(row["image_id"])
        image_name = Path(str(row["image_name"])).name   # remove folder prefix if present
        image_path = Path(cfg.TEST_IMAGE_DIR) / image_name

        if image_path.exists():
            candidate_image_ids.append(image_id)

    print("Total image_pool ids     :", len(image_pool_df))
    print("Available test image ids :", len(candidate_image_ids))

    img_ids, img_embs = compute_image_embeddings(
        model,
        candidate_image_ids,
        cfg.TEST_IMAGE_DIR
    )

    # --------------------------------------------------
    # 3. Split test queries
    # --------------------------------------------------
    test_image_queries = test_df[test_df["query_type"] == "image"].copy()
    test_caption_queries = test_df[test_df["query_type"] == "caption"].copy()

    rows = []

    # --------------------------------------------------
    # 4. IMAGE -> TEXT
    # --------------------------------------------------
    q_img_ids, q_img_embs = compute_image_embeddings(
        model,
        test_image_queries["query"].astype(int).tolist(),
        cfg.TEST_IMAGE_DIR
    )

    sim_i2t = q_img_embs @ cap_embs.T
    top10_i2t = torch.topk(sim_i2t, k=10, dim=1).indices.numpy()
    pred_caps = [[cap_ids[j] for j in row] for row in top10_i2t]

    for row, preds in zip(test_image_queries.itertuples(index=False), pred_caps):
        rows.append({
            "id": row.id,
            **{f"top{i+1}": preds[i] for i in range(10)}
        })

    # --------------------------------------------------
    # 5. CAPTION -> IMAGE
    # --------------------------------------------------
    q_cap_ids, q_cap_embs = compute_text_embeddings(
        model,
        test_caption_queries["query"].astype(int).tolist()
    )

    sim_t2i = q_cap_embs @ img_embs.T
    top10_t2i = torch.topk(sim_t2i, k=10, dim=1).indices.numpy()
    pred_imgs = [[img_ids[j] for j in row] for row in top10_t2i]

    for row, preds in zip(test_caption_queries.itertuples(index=False), pred_imgs):
        rows.append({
            "id": row.id,
            **{f"top{i+1}": preds[i] for i in range(10)}
        })

    # --------------------------------------------------
    # 6. Build submission in correct order
    # --------------------------------------------------
    sub = pd.DataFrame(rows)
    sub = sample_submission_df[["id"]].merge(sub, on="id", how="left")

    return sub

In [52]:
from pathlib import Path

image_pool_df["image_name"] = image_pool_df["image_name"].astype(str).apply(lambda x: Path(x).name)
image_id_to_name = dict(zip(image_pool_df["image_id"], image_pool_df["image_name"]))

In [53]:
submission_base = build_base_submission(model)
base_path = Path(cfg.OUTPUT_DIR) / "submission_lora_base.csv"
submission_base.to_csv(base_path, index=False)
print(base_path)
display(submission_base.head())

Encode texts:   0%|          | 0/391 [00:00<?, ?it/s]

Total image_pool ids     : 10000
Available test image ids : 1000


Encode images:   0%|          | 0/8 [00:00<?, ?it/s]

Encode images:   0%|          | 0/4 [00:00<?, ?it/s]

Encode texts:   0%|          | 0/4 [00:00<?, ?it/s]

outputs_lora_clip/submission_lora_base.csv


,id,top1,top2,top3,top4,top5,top6,top7,top8,top9,top10
0,1,31151,43297,47109,43215,24252,47108,34143,20567,47373,34144
1,2,49396,49397,45384,13183,40688,49166,49399,49398,49168,3096
2,3,48001,48003,14673,48002,48004,48005,6183,26892,26895,6185
3,4,20233,49900,22235,18649,49896,36020,49085,18646,47555,20235
4,5,46616,46618,46620,46617,46619,39466,36563,35581,45928,7650


## Lightweight rerank

This rerank block is intentionally simple and notebook-friendly.
It reranks the base top-10 using the same CLIP scores again so the structure is easy to replace later with BLIP or another cross-encoder.

In [56]:
@torch.no_grad()
def rerank_submission_with_clip(model, base_submission):
    from pathlib import Path

    model.eval()

    # --------------------------------------------------
    # 1. Caption candidate pool
    # --------------------------------------------------
    candidate_caption_ids = caption_pool_df["caption_id"].astype(int).tolist()
    cap_ids, cap_embs = compute_text_embeddings(model, candidate_caption_ids)

    # --------------------------------------------------
    # 2. Image candidate pool
    #    Keep only images that actually exist in test_images
    # --------------------------------------------------
    candidate_image_ids = []
    for _, row in image_pool_df.iterrows():
        image_id = int(row["image_id"])
        image_name = Path(str(row["image_name"])).name
        image_path = Path(cfg.TEST_IMAGE_DIR) / image_name

        if image_path.exists():
            candidate_image_ids.append(image_id)

    print("Total image_pool ids     :", len(image_pool_df))
    print("Available test image ids :", len(candidate_image_ids))

    img_ids, img_embs = compute_image_embeddings(
        model,
        candidate_image_ids,
        cfg.TEST_IMAGE_DIR
    )

    # --------------------------------------------------
    # 3. Build fast lookup maps
    # --------------------------------------------------
    cap_id_to_idx = {cid: i for i, cid in enumerate(cap_ids)}
    img_id_to_idx = {iid: i for i, iid in enumerate(img_ids)}

    # --------------------------------------------------
    # 4. Split test queries
    # --------------------------------------------------
    test_image_queries = test_df[test_df["query_type"] == "image"].copy()
    test_caption_queries = test_df[test_df["query_type"] == "caption"].copy()

    rows = []

    # --------------------------------------------------
    # 5. IMAGE -> TEXT reranking
    # --------------------------------------------------
    for row in tqdm(
        test_image_queries.itertuples(index=False),
        total=len(test_image_queries),
        desc="Rerank image queries"
    ):
        query_image_id = int(row.query)

        # encode query image
        _, q_emb = compute_image_embeddings(model, [query_image_id], cfg.TEST_IMAGE_DIR)
        q_emb = q_emb[0]

        # read existing top-10 caption candidates from base submission
        candidate_ids = [
            int(base_submission.loc[base_submission["id"] == row.id, f"top{i}"].values[0])
            for i in range(1, 11)
        ]

        # keep only those candidates that exist in caption pool
        candidate_ids = [cid for cid in candidate_ids if cid in cap_id_to_idx]

        cand_idx = [cap_id_to_idx[cid] for cid in candidate_ids]
        cand_embs = cap_embs[cand_idx]

        scores = (q_emb[None, :] @ cand_embs.T).squeeze(0)
        order = torch.argsort(scores, descending=True).tolist()
        preds = [candidate_ids[i] for i in order]

        # safety padding if something was filtered out
        while len(preds) < 10:
            preds.append(preds[-1] if preds else cap_ids[0])

        rows.append({
            "id": row.id,
            **{f"top{i+1}": preds[i] for i in range(10)}
        })

    # --------------------------------------------------
    # 6. CAPTION -> IMAGE reranking
    # --------------------------------------------------
    for row in tqdm(
        test_caption_queries.itertuples(index=False),
        total=len(test_caption_queries),
        desc="Rerank caption queries"
    ):
        query_caption_id = int(row.query)

        # encode query caption
        _, q_emb = compute_text_embeddings(model, [query_caption_id])
        q_emb = q_emb[0]

        # read existing top-10 image candidates from base submission
        candidate_ids = [
            int(base_submission.loc[base_submission["id"] == row.id, f"top{i}"].values[0])
            for i in range(1, 11)
        ]

        # keep only candidates that actually exist in img_ids
        candidate_ids = [iid for iid in candidate_ids if iid in img_id_to_idx]

        cand_idx = [img_id_to_idx[iid] for iid in candidate_ids]
        cand_embs = img_embs[cand_idx]

        scores = (q_emb[None, :] @ cand_embs.T).squeeze(0)
        order = torch.argsort(scores, descending=True).tolist()
        preds = [candidate_ids[i] for i in order]

        # safety padding if something was filtered out
        while len(preds) < 10:
            preds.append(preds[-1] if preds else img_ids[0])

        rows.append({
            "id": row.id,
            **{f"top{i+1}": preds[i] for i in range(10)}
        })

    # --------------------------------------------------
    # 7. Build submission in correct order
    # --------------------------------------------------
    sub = pd.DataFrame(rows)
    sub = sample_submission_df[["id"]].merge(sub, on="id", how="left")
    return sub

In [57]:
submission_rerank = rerank_submission_with_clip(model, submission_base)
rerank_path = Path(cfg.OUTPUT_DIR) / "submission_lora_rerank.csv"
submission_rerank.to_csv(rerank_path, index=False)
print(rerank_path)
display(submission_rerank.head())

Encode texts:   0%|          | 0/391 [00:00<?, ?it/s]

Total image_pool ids     : 10000
Available test image ids : 1000


Encode images:   0%|          | 0/8 [00:00<?, ?it/s]

Rerank image queries:   0%|          | 0/500 [00:00<?, ?it/s]

Encode images:   0%|          | 0/1 [00:00<?, ?it/s]

...

Encode texts:   0%|          | 0/1 [00:00<?, ?it/s]

Encode texts:   0%|          | 0/1 [00:00<?, ?it/s]

Encode texts:   0%|          | 0/1 [00:00<?, ?it/s]

outputs_lora_clip/submission_lora_rerank.csv


,id,top1,top2,top3,top4,top5,top6,top7,top8,top9,top10
0,1,31151,43297,47109,43215,24252,47108,34143,20567,47373,34144
1,2,49396,49397,45384,13183,40688,49166,49399,49398,49168,3096
2,3,48001,48003,14673,48002,48004,48005,6183,26892,26895,6185
3,4,20233,49900,22235,18649,49896,36020,49085,18646,47555,20235
4,5,46616,46618,46620,46617,46619,39466,36563,35581,45928,7650


In [58]:
print(rerank_path)
display(submission_rerank.head())

outputs_lora_clip/submission_lora_rerank.csv


,id,top1,top2,top3,top4,top5,top6,top7,top8,top9,top10
0,1,31151,43297,47109,43215,24252,47108,34143,20567,47373,34144
1,2,49396,49397,45384,13183,40688,49166,49399,49398,49168,3096
2,3,48001,48003,14673,48002,48004,48005,6183,26892,26895,6185
3,4,20233,49900,22235,18649,49896,36020,49085,18646,47555,20235
4,5,46616,46618,46620,46617,46619,39466,36563,35581,45928,7650


In [59]:
@torch.no_grad()
def build_top50_candidates(model, retrieve_k=50):
    from pathlib import Path

    model.eval()

    candidate_caption_ids = caption_pool_df["caption_id"].astype(int).tolist()
    cap_ids, cap_embs = compute_text_embeddings(model, candidate_caption_ids)

    candidate_image_ids = []
    for _, row in image_pool_df.iterrows():
        image_id = int(row["image_id"])
        image_name = Path(str(row["image_name"])).name
        image_path = Path(cfg.TEST_IMAGE_DIR) / image_name
        if image_path.exists():
            candidate_image_ids.append(image_id)

    img_ids, img_embs = compute_image_embeddings(
        model,
        candidate_image_ids,
        cfg.TEST_IMAGE_DIR
    )

    test_image_queries = test_df[test_df["query_type"] == "image"].copy()
    test_caption_queries = test_df[test_df["query_type"] == "caption"].copy()

    top50_candidates = {}

    # image -> caption top-50
    q_img_ids, q_img_embs = compute_image_embeddings(
        model,
        test_image_queries["query"].astype(int).tolist(),
        cfg.TEST_IMAGE_DIR
    )
    sim_i2t = q_img_embs @ cap_embs.T
    top50_i2t = torch.topk(sim_i2t, k=min(retrieve_k, len(cap_ids)), dim=1).indices.numpy()

    for row, idxs in zip(test_image_queries.itertuples(index=False), top50_i2t):
        top50_candidates[row.id] = {
            "caption_candidates": [cap_ids[j] for j in idxs]
        }

    # caption -> image top-50
    q_cap_ids, q_cap_embs = compute_text_embeddings(
        model,
        test_caption_queries["query"].astype(int).tolist()
    )
    sim_t2i = q_cap_embs @ img_embs.T
    top50_t2i = torch.topk(sim_t2i, k=min(retrieve_k, len(img_ids)), dim=1).indices.numpy()

    for row, idxs in zip(test_caption_queries.itertuples(index=False), top50_t2i):
        top50_candidates[row.id] = {
            "image_candidates": [img_ids[j] for j in idxs]
        }

    return top50_candidates

In [61]:
@torch.no_grad()
def rerank_submission_with_clip(model, top50_candidates, final_k=10):
    from pathlib import Path

    model.eval()

    # --------------------------------------------------
    # 1. Caption candidate pool
    # --------------------------------------------------
    candidate_caption_ids = caption_pool_df["caption_id"].astype(int).tolist()
    cap_ids, cap_embs = compute_text_embeddings(model, candidate_caption_ids)

    # --------------------------------------------------
    # 2. Image candidate pool
    #    Keep only images that actually exist in test_images
    # --------------------------------------------------
    candidate_image_ids = []
    for _, row in image_pool_df.iterrows():
        image_id = int(row["image_id"])
        image_name = Path(str(row["image_name"])).name
        image_path = Path(cfg.TEST_IMAGE_DIR) / image_name

        if image_path.exists():
            candidate_image_ids.append(image_id)

    print("Total image_pool ids     :", len(image_pool_df))
    print("Available test image ids :", len(candidate_image_ids))

    img_ids, img_embs = compute_image_embeddings(
        model,
        candidate_image_ids,
        cfg.TEST_IMAGE_DIR
    )

    # --------------------------------------------------
    # 3. Build fast lookup maps
    # --------------------------------------------------
    cap_id_to_idx = {cid: i for i, cid in enumerate(cap_ids)}
    img_id_to_idx = {iid: i for i, iid in enumerate(img_ids)}

    # --------------------------------------------------
    # 4. Split test queries
    # --------------------------------------------------
    test_image_queries = test_df[test_df["query_type"] == "image"].copy()
    test_caption_queries = test_df[test_df["query_type"] == "caption"].copy()

    rows = []

    # --------------------------------------------------
    # 5. IMAGE -> TEXT reranking from top-50
    # --------------------------------------------------
    for row in tqdm(
        test_image_queries.itertuples(index=False),
        total=len(test_image_queries),
        desc="Rerank image queries"
    ):
        query_image_id = int(row.query)

        _, q_emb = compute_image_embeddings(model, [query_image_id], cfg.TEST_IMAGE_DIR)
        q_emb = q_emb[0]

        # read top-50 caption candidates
        candidate_ids = top50_candidates[row.id]["caption_candidates"]
        candidate_ids = [cid for cid in candidate_ids if cid in cap_id_to_idx]

        cand_idx = [cap_id_to_idx[cid] for cid in candidate_ids]
        cand_embs = cap_embs[cand_idx]

        scores = (q_emb[None, :] @ cand_embs.T).squeeze(0)
        order = torch.argsort(scores, descending=True).tolist()
        preds = [candidate_ids[i] for i in order[:final_k]]

        while len(preds) < final_k:
            preds.append(preds[-1] if preds else cap_ids[0])

        rows.append({
            "id": row.id,
            **{f"top{i+1}": preds[i] for i in range(final_k)}
        })

    # --------------------------------------------------
    # 6. CAPTION -> IMAGE reranking from top-50
    # --------------------------------------------------
    for row in tqdm(
        test_caption_queries.itertuples(index=False),
        total=len(test_caption_queries),
        desc="Rerank caption queries"
    ):
        query_caption_id = int(row.query)

        _, q_emb = compute_text_embeddings(model, [query_caption_id])
        q_emb = q_emb[0]

        # read top-50 image candidates
        candidate_ids = top50_candidates[row.id]["image_candidates"]
        candidate_ids = [iid for iid in candidate_ids if iid in img_id_to_idx]

        cand_idx = [img_id_to_idx[iid] for iid in candidate_ids]
        cand_embs = img_embs[cand_idx]

        scores = (q_emb[None, :] @ cand_embs.T).squeeze(0)
        order = torch.argsort(scores, descending=True).tolist()
        preds = [candidate_ids[i] for i in order[:final_k]]

        while len(preds) < final_k:
            preds.append(preds[-1] if preds else img_ids[0])

        rows.append({
            "id": row.id,
            **{f"top{i+1}": preds[i] for i in range(final_k)}
        })

    # --------------------------------------------------
    # 7. Build final submission
    # --------------------------------------------------
    sub = pd.DataFrame(rows)
    sub = sample_submission_df[["id"]].merge(sub, on="id", how="left")
    return sub

In [62]:
@torch.no_grad()
def build_top50_candidates(model, retrieve_k=50):
    from pathlib import Path

    model.eval()

    candidate_caption_ids = caption_pool_df["caption_id"].astype(int).tolist()
    cap_ids, cap_embs = compute_text_embeddings(model, candidate_caption_ids)

    candidate_image_ids = []
    for _, row in image_pool_df.iterrows():
        image_id = int(row["image_id"])
        image_name = Path(str(row["image_name"])).name
        image_path = Path(cfg.TEST_IMAGE_DIR) / image_name
        if image_path.exists():
            candidate_image_ids.append(image_id)

    img_ids, img_embs = compute_image_embeddings(
        model,
        candidate_image_ids,
        cfg.TEST_IMAGE_DIR
    )

    test_image_queries = test_df[test_df["query_type"] == "image"].copy()
    test_caption_queries = test_df[test_df["query_type"] == "caption"].copy()

    top50_candidates = {}

    # image -> caption top-50
    q_img_ids, q_img_embs = compute_image_embeddings(
        model,
        test_image_queries["query"].astype(int).tolist(),
        cfg.TEST_IMAGE_DIR
    )
    sim_i2t = q_img_embs @ cap_embs.T
    top50_i2t = torch.topk(sim_i2t, k=min(retrieve_k, len(cap_ids)), dim=1).indices.numpy()

    for row, idxs in zip(test_image_queries.itertuples(index=False), top50_i2t):
        top50_candidates[row.id] = {
            "caption_candidates": [cap_ids[j] for j in idxs]
        }

    # caption -> image top-50
    q_cap_ids, q_cap_embs = compute_text_embeddings(
        model,
        test_caption_queries["query"].astype(int).tolist()
    )
    sim_t2i = q_cap_embs @ img_embs.T
    top50_t2i = torch.topk(sim_t2i, k=min(retrieve_k, len(img_ids)), dim=1).indices.numpy()

    for row, idxs in zip(test_caption_queries.itertuples(index=False), top50_t2i):
        top50_candidates[row.id] = {
            "image_candidates": [img_ids[j] for j in idxs]
        }

    return top50_candidates

In [63]:
top50_candidates = build_top50_candidates(model, retrieve_k=50)

submission_rerank_50 = rerank_submission_with_clip(
    model,
    top50_candidates=top50_candidates,
    final_k=10
)

rerank50_path = Path(cfg.OUTPUT_DIR) / "submission_lora_rerank_top50.csv"
submission_rerank_50.to_csv(rerank50_path, index=False)

print(rerank50_path)
display(submission_rerank_50.head())

Encode texts:   0%|          | 0/391 [00:00<?, ?it/s]

Encode images:   0%|          | 0/8 [00:00<?, ?it/s]

Encode images:   0%|          | 0/4 [00:00<?, ?it/s]

Encode texts:   0%|          | 0/4 [00:00<?, ?it/s]

Encode texts:   0%|          | 0/391 [00:00<?, ?it/s]

...

Encode texts:   0%|          | 0/1 [00:00<?, ?it/s]

Encode texts:   0%|          | 0/1 [00:00<?, ?it/s]

Encode texts:   0%|          | 0/1 [00:00<?, ?it/s]

outputs_lora_clip/submission_lora_rerank_top50.csv


,id,top1,top2,top3,top4,top5,top6,top7,top8,top9,top10
0,1,31151,43297,47109,43215,24252,47108,34143,20567,47373,34144
1,2,49396,49397,45384,13183,40688,49166,49399,49398,49168,3096
2,3,48001,48003,14673,48002,48004,48005,6183,26892,26895,6185
3,4,20233,49900,22235,18649,49896,36020,49085,18646,47555,20235
4,5,46616,46618,46620,46617,46619,39466,36563,35581,45928,7650


In [64]:
print(rerank50_path)
display(submission_rerank_50.head())

outputs_lora_clip/submission_lora_rerank_top50.csv


,id,top1,top2,top3,top4,top5,top6,top7,top8,top9,top10
0,1,31151,43297,47109,43215,24252,47108,34143,20567,47373,34144
1,2,49396,49397,45384,13183,40688,49166,49399,49398,49168,3096
2,3,48001,48003,14673,48002,48004,48005,6183,26892,26895,6185
3,4,20233,49900,22235,18649,49896,36020,49085,18646,47555,20235
4,5,46616,46618,46620,46617,46619,39466,36563,35581,45928,7650
